In [0]:
from pyspark.sql.functions import avg, when, col


In [0]:
silver_sales_path = "/Volumes/workspace/m5_project/silver/silver_sales"
silver_prices_path = "/Volumes/workspace/m5_project/silver/silver_prices"

silver_sales_df = spark.read.parquet(
    silver_sales_path
)

silver_prices_df = spark.read.parquet(
    silver_prices_path
)

display(silver_sales_df.limit(5))
display(silver_prices_df.limit(5))

id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk
HOUSEHOLD_1_363_TX_3_validation,HOUSEHOLD_1_363,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_364_TX_3_validation,HOUSEHOLD_1_364,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_365_TX_3_validation,HOUSEHOLD_1_365,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_366_TX_3_validation,HOUSEHOLD_1_366,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101
HOUSEHOLD_1_367_TX_3_validation,HOUSEHOLD_1_367,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,0,2011-01-29,11101


store_id,item_id,wm_yr_wk,sell_price
CA_4,FOODS_3_204,11514,2.48
CA_4,FOODS_3_204,11515,2.48
CA_4,FOODS_3_204,11516,2.48
CA_4,FOODS_3_204,11517,2.48
CA_4,FOODS_3_204,11518,2.48


In [0]:
print(silver_sales_df.columns)

['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'day', 'sales', 'date', 'wm_yr_wk']


In [0]:
# weekly_sales_df = silver_sales_df.groupBy(
#     "item_id",
#     "store_id",
#     "wm_yr_wk"
# ).agg(
#     avg("sales").alias("avg_weekly_sales")
# )

# display(weekly_sales_df.limit(10))

weekly_sales_df = silver_sales_df.groupBy(
    "item_id",
    "store_id",
    "cat_id",
    "dept_id",
    "state_id",
    "wm_yr_wk"
).agg(
    avg("sales").alias("avg_weekly_sales")
)

display(weekly_sales_df.limit(10))

item_id,store_id,cat_id,dept_id,state_id,wm_yr_wk,avg_weekly_sales
HOUSEHOLD_1_364,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,11101,0.0
HOUSEHOLD_1_457,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,11101,0.0
HOUSEHOLD_1_499,TX_3,HOUSEHOLD,HOUSEHOLD_1,TX,11101,0.0
HOUSEHOLD_2_021,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,11101,0.42857142857142855
HOUSEHOLD_2_108,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,11101,0.0
HOUSEHOLD_2_252,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,11101,0.0
HOUSEHOLD_2_269,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,11101,0.0
HOUSEHOLD_2_319,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,11101,0.0
HOUSEHOLD_2_356,TX_3,HOUSEHOLD,HOUSEHOLD_2,TX,11101,0.8571428571428571
FOODS_2_078,TX_3,FOODS,FOODS_2,TX,11101,0.0


#join that with prices

In [0]:
price_impact_df = weekly_sales_df.join(
    silver_prices_df,
    on=[
        "item_id",
        "store_id",
        "wm_yr_wk"
    ],
    how="inner"
)

display(price_impact_df.limit(10))

item_id,store_id,wm_yr_wk,cat_id,dept_id,state_id,avg_weekly_sales,sell_price
HOUSEHOLD_1_379,TX_3,11101,HOUSEHOLD,HOUSEHOLD_1,TX,5.857142857142857,0.97
HOUSEHOLD_2_155,TX_3,11101,HOUSEHOLD,HOUSEHOLD_2,TX,0.14285714285714285,6.47
HOUSEHOLD_2_376,TX_3,11101,HOUSEHOLD,HOUSEHOLD_2,TX,0.5714285714285714,3.27
FOODS_3_178,TX_3,11101,FOODS,FOODS_3,TX,2.4285714285714284,1.52
FOODS_3_184,TX_3,11101,FOODS,FOODS_3,TX,1.1428571428571428,2.98
FOODS_3_219,TX_3,11101,FOODS,FOODS_3,TX,5.857142857142857,1.0
FOODS_3_248,TX_3,11101,FOODS,FOODS_3,TX,2.2857142857142856,1.98
FOODS_3_784,TX_3,11101,FOODS,FOODS_3,TX,5.428571428571429,1.0
HOUSEHOLD_1_138,WI_1,11101,HOUSEHOLD,HOUSEHOLD_1,WI,1.4285714285714286,4.97
HOUSEHOLD_2_010,WI_1,11101,HOUSEHOLD,HOUSEHOLD_2,WI,0.14285714285714285,6.97


In [0]:
price_impact_df = (
    price_impact_df
    .withColumn("category", col("cat_id"))
    .withColumn("department", col("dept_id"))
)



In [0]:
#evaluate for price sensitivity

price_impact_df = price_impact_df.withColumn(
    "price_sensitivity_flag",
    when(
        (col("sell_price") < 2) &
        (col("avg_weekly_sales") > 5),
        "HIGH"
    ).when(
        col("avg_weekly_sales") > 2,
        "MEDIUM"
    ).otherwise(
        "LOW"
    )
)

display(price_impact_df.limit(20))

item_id,store_id,wm_yr_wk,cat_id,dept_id,state_id,avg_weekly_sales,sell_price,category,department,price_sensitivity_flag
HOUSEHOLD_1_379,TX_3,11101,HOUSEHOLD,HOUSEHOLD_1,TX,5.857142857142857,0.97,HOUSEHOLD,HOUSEHOLD_1,HIGH
HOUSEHOLD_2_155,TX_3,11101,HOUSEHOLD,HOUSEHOLD_2,TX,0.14285714285714285,6.47,HOUSEHOLD,HOUSEHOLD_2,LOW
HOUSEHOLD_2_376,TX_3,11101,HOUSEHOLD,HOUSEHOLD_2,TX,0.5714285714285714,3.27,HOUSEHOLD,HOUSEHOLD_2,LOW
FOODS_3_178,TX_3,11101,FOODS,FOODS_3,TX,2.4285714285714284,1.52,FOODS,FOODS_3,MEDIUM
FOODS_3_184,TX_3,11101,FOODS,FOODS_3,TX,1.1428571428571428,2.98,FOODS,FOODS_3,LOW
FOODS_3_219,TX_3,11101,FOODS,FOODS_3,TX,5.857142857142857,1.0,FOODS,FOODS_3,HIGH
FOODS_3_248,TX_3,11101,FOODS,FOODS_3,TX,2.2857142857142856,1.98,FOODS,FOODS_3,MEDIUM
FOODS_3_784,TX_3,11101,FOODS,FOODS_3,TX,5.428571428571429,1.0,FOODS,FOODS_3,HIGH
HOUSEHOLD_1_138,WI_1,11101,HOUSEHOLD,HOUSEHOLD_1,WI,1.4285714285714286,4.97,HOUSEHOLD,HOUSEHOLD_1,LOW
HOUSEHOLD_2_010,WI_1,11101,HOUSEHOLD,HOUSEHOLD_2,WI,0.14285714285714285,6.97,HOUSEHOLD,HOUSEHOLD_2,LOW


In [0]:
gold_price_path = "/Volumes/workspace/m5_project/gold/gold_price_impact"

price_impact_df.write.mode("overwrite").parquet(
    gold_price_path
)

print("Gold Price Impact written successfully")

Gold Price Impact written successfully


In [0]:
gold_price_check = spark.read.parquet(
    gold_price_path
)

display(gold_price_check.limit(20))

item_id,store_id,wm_yr_wk,cat_id,dept_id,state_id,avg_weekly_sales,sell_price,category,department,price_sensitivity_flag
FOODS_1_136,TX_3,11101,FOODS,FOODS_1,TX,0.14285714285714285,3.97,FOODS,FOODS_1,LOW
FOODS_1_219,TX_3,11101,FOODS,FOODS_1,TX,1.1428571428571428,2.0,FOODS,FOODS_1,LOW
FOODS_3_377,TX_3,11101,FOODS,FOODS_3,TX,32.57142857142857,1.38,FOODS,FOODS_3,HIGH
FOODS_3_793,TX_3,11101,FOODS,FOODS_3,TX,0.42857142857142855,2.98,FOODS,FOODS_3,LOW
HOBBIES_1_194,WI_1,11101,HOBBIES,HOBBIES_1,WI,2.5714285714285716,2.5,HOBBIES,HOBBIES_1,MEDIUM
HOUSEHOLD_1_230,WI_1,11101,HOUSEHOLD,HOUSEHOLD_1,WI,1.0,2.92,HOUSEHOLD,HOUSEHOLD_1,LOW
HOUSEHOLD_1_345,WI_1,11101,HOUSEHOLD,HOUSEHOLD_1,WI,0.7142857142857143,3.72,HOUSEHOLD,HOUSEHOLD_1,LOW
HOUSEHOLD_2_047,WI_1,11101,HOUSEHOLD,HOUSEHOLD_2,WI,0.42857142857142855,2.97,HOUSEHOLD,HOUSEHOLD_2,LOW
FOODS_2_107,TX_3,11102,FOODS,FOODS_2,TX,0.14285714285714285,1.88,FOODS,FOODS_2,LOW
FOODS_2_342,TX_3,11102,FOODS,FOODS_2,TX,1.0,5.86,FOODS,FOODS_2,LOW


In [0]:
print(silver_sales_df.columns)

['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'day', 'sales', 'date', 'wm_yr_wk']


In [0]:
gold_price_export_path = "/Volumes/workspace/m5_project/gold/exports/gold_price_impact_csv"

price_impact_df.write.mode("overwrite") \
    .option("header", True) \
    .csv(gold_price_export_path)

print("Price Impact CSV export complete")

Price Impact CSV export complete


In [0]:
silver_sales_path = "/Volumes/workspace/m5_project/silver/silver_sales"
silver_prices_path = "/Volumes/workspace/m5_project/silver/silver_prices"

silver_sales_df = spark.read.parquet(silver_sales_path)
silver_prices_df = spark.read.parquet(silver_prices_path)

In [0]:
print(silver_sales_df.count())
print(silver_prices_df.count())
print(price_impact_df.count())

58327370
6841121
6597201


In [0]:
# price_impact_df = silver_sales_df.join(
#     silver_prices_df,
#     on=["item_id", "store_id", "wm_yr_wk"],
#     how="inner"
# )

# print("Price impact dataframe created")

Price impact dataframe created
